# Juego del Gato con MINIMAX

**Actividad:** implementar el algoritmo *minimax* aplicado al juego del gato o tres en linea
(3x3), de modo que una persona pueda jugar contra la computadora.

---

## Que exige la actividad

| # | Requisito | Donde se cumple |
|---|-----------|-----------------|
| 1 | Una persona juega contra la computadora | Seccion 7 (texto) y 8 (grafica) |
| 2 | Solo se permiten movimientos validos | `jugar()` valida y lanza `ValueError` |
| 3 | Se respetan los turnos de cada jugador | Los bucles de juego alternan estrictamente |
| 4 | La computadora **nunca pierde** | Demostrado en la seccion 9 |

## Como usar este notebook

Ejecuta las celdas **en orden** (`Cell > Run All`, o `Ctrl+Enter` una por una).

* Las secciones 1 a 6 son la implementacion y sus pruebas: no necesitan tu intervencion.
* La **seccion 7** te deja jugar por consola (escribes el numero de la casilla).
* La **seccion 8** abre un tablero grafico con botones (requiere `ipywidgets`).
* La **seccion 9** es la demostracion automatica de que la computadora no pierde.

> Si estas en Google Colab y la seccion 8 falla, ejecuta primero
> `!pip install ipywidgets` en una celda nueva.

In [ ]:
# Dependencias: la notebook solo usa la libreria estandar.
# ipywidgets es OPCIONAL y se usa unicamente en la seccion 8 (interfaz grafica).
try:
    import ipywidgets
    WIDGETS_DISPONIBLES = True
except ImportError:
    WIDGETS_DISPONIBLES = False

import time
from functools import lru_cache

print("Python listo.")
print("ipywidgets disponible:", WIDGETS_DISPONIBLES)

---
## 1. Representacion del tablero

El tablero es una **tupla de 9 elementos**, uno por casilla, en orden de lectura:

```
indice:  0 | 1 | 2
         3 | 4 | 5
         6 | 7 | 8
```

Cada casilla vale `VACIO` (`None`), `X` u `O`. Se usa una tupla y no una lista porque es
**inmutable y hashable**, lo que permite cachear los resultados del minimax con `lru_cache`.

In [ ]:
VACIO = None
X = "X"
O = "O"

# Las 8 lineas que dan la victoria (3 filas, 3 columnas, 2 diagonales).
LINEAS = (
    (0, 1, 2), (3, 4, 5), (6, 7, 8),      # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),      # columnas
    (0, 4, 8), (2, 4, 6),                 # diagonales
)

TABLERO_VACIO = (VACIO,) * 9

SIMBOLOS = {VACIO: " ", X: "X", O: "O"}


def el_otro(simbolo):
    """Devuelve el simbolo rival de `simbolo`."""
    return O if simbolo == X else X


def tablero_texto(tablero, numerar=False, resaltar=None):
    """Representa el tablero como texto. `resaltar` marca una casilla con asteriscos."""
    lineas = []
    for fila in range(3):
        celdas = []
        for col in range(3):
            i = fila * 3 + col
            marca = SIMBOLOS[tablero[i]]
            if numerar and tablero[i] is VACIO:
                marca = str(i + 1)
            celdas.append(f" {marca} " if i != resaltar else f"*{marca}*")
        lineas.append(" | ".join(celdas))
        if fila < 2:
            lineas.append("---+---+---")
    return "\n".join(lineas)


print(tablero_texto(TABLERO_VACIO))
print()
print(tablero_texto(TABLERO_VACIO, numerar=True))
print()
print("Ultima casilla (indice 8):", repr(TABLERO_VACIO[8]))
print("Tipo del tablero:", type(TABLERO_VACIO).__name__, "| es hashable:", isinstance(hash(TABLERO_VACIO), int))

---
## 2. Reglas del juego

`jugar()` es la **unica** forma de modificar el tablero. Concentra las validaciones para que
ningun punto del codigo pueda saltarse una regla.

In [ ]:
@lru_cache(maxsize=None)
def ganador(tablero):
    """Devuelve el simbolo ganador, o VACIO si todavia no hay linea."""
    for a, b, c in LINEAS:
        if tablero[a] is not VACIO and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return VACIO


@lru_cache(maxsize=None)
def es_empate(tablero):
    """True si el tablero se lleno y nadie gano."""
    return VACIO not in tablero and ganador(tablero) is VACIO


@lru_cache(maxsize=None)
def es_terminal(tablero):
    """True si la partida termino (hay ganador o empate)."""
    return ganador(tablero) is not VACIO or es_empate(tablero)


@lru_cache(maxsize=None)
def movimientos_validos(tablero):
    """Casillas libres, en orden. Es el unico origen de movimientos legitimos."""
    if es_terminal(tablero):
        return ()
    return tuple(i for i in range(9) if tablero[i] is VACIO)


def jugar(tablero, movimiento, simbolo):
    """Coloca `simbolo` en `movimiento` y devuelve un tablero NUEVO.

    Lanza ValueError si el movimiento es invalido o si la partida ya termino.
    """
    if simbolo not in (X, O):
        raise ValueError(f"Simbolo invalido: {simbolo!r}")
    if es_terminal(tablero):
        raise ValueError("La partida ya termino, no se puede seguir jugando.")
    if movimiento not in movimientos_validos(tablero):
        raise ValueError(f"Movimiento invalido: la casilla {movimiento} no esta libre.")
    return tablero[:movimiento] + (simbolo,) + tablero[movimiento + 1:]

In [ ]:
# --- Pruebas de las reglas ---
ejemplos = [
    (TABLERO_VACIO, "tablero vacio"),
    (jugar(TABLERO_VACIO, 4, X), "X en el centro"),
    (jugar(jugar(TABLERO_VACIO, 0, X), 8, X), "diagonal de X"),
]

for t, descripcion in ejemplos:
    g = ganador(t)
    print(f"{descripcion:22s} ganador={g or '-'}  empate={es_empate(t)}  "
          f"terminal={es_terminal(t)}  libres={movimientos_validos(t)}")

print()

# El tablero original NO se modifica: jugar() devuelve una copia.
base = jugar(TABLERO_VACIO, 0, X)
copia = jugar(base, 1, O)
print("inmutabilidad -> original:", tablero_texto(base).replace(chr(10), " / "))
print("inmutabilidad -> copia   :", tablero_texto(copia).replace(chr(10), " / "))

print()

# Validaciones: los movimientos invalidos deben ser rechazados.
partida_ganada = jugar(jugar(jugar(TABLERO_VACIO, 0, X), 4, X), 8, X)  # diagonal de X
errores = [
    ("casilla ocupada", lambda: jugar(base, 0, O)),
    ("indice fuera de rango", lambda: jugar(TABLERO_VACIO, 12, X)),
    ("movimiento negativo", lambda: jugar(TABLERO_VACIO, -1, X)),
    ("simbolo desconocido", lambda: jugar(TABLERO_VACIO, 4, "Z")),
    ("tras terminar la partida", lambda: jugar(partida_ganada, 6, O)),
]
for descripcion, accion in errores:
    try:
        accion()
        print(f"  {descripcion:24s} -> *** NO SE RECHAZO ***")
    except ValueError as e:
        print(f"  {descripcion:24s} -> rechazado: {e}")

---
## 3. Algoritmo MINIMAX

### Idea

Minimax construye el **arbol de juego completo** y lo etiqueta de abajo hacia arriba. Cada
nodo es un tablero y cada arista es un movimiento legal. La evaluacion es jerarquica:

* En los nodos del **MAX** (la computadora) se queda con el **maximo** de los hijos.
* En los nodos del **MIN** (la persona) se queda con el **minimo** de los hijos.

Si un MAX elige entre un hijo que le da `10` y otro que le da `-10`, jamas eligira el segundo,
porque sabe que en ese subarbol MIN siempre podria tomarlo.

### Asignacion de valores

El valor de una hoja se mide **desde el punto de vista de la computadora**:

| Situacion | Valor | Razon |
|-----------|-------|-------|
| Gano la computadora | `10 - profundidad` | Positivo: penaliza ganar tarde frente a ganar rapido |
| Gano la persona | `profundidad - 10` | Negativo: prefiere perder lo mas tarde posible |
| Empate | `0` | Neutro |
| La partida sigue | MAX toma el max de los hijos; MIN toma el min | Decision racional |

El termino `profundidad` es lo que hace que la computadora **no se conforme** con una victoria
segura si puede conseguirla antes. `profundidad` cuenta los movimientos ya jugados.

### Parametros

`simbolo_max` y `simbolo_min` se mantienen **fijos** en toda la recursion. Lo unico que
alterna es la bandera `maximizing` y, por tanto, que simbolo se coloca en el tablero. Asi la
puntuacion siempre es comparable entre ramas.

In [ ]:
INFINITO = float("inf")

# Contadores de nodos, para medir cuanto recorrido hace cada algoritmo.
NODOS_MINIMAX = 0
NODOS_ALFA_BETA = 0


def reiniciar_contadores():
    global NODOS_MINIMAX, NODOS_ALFA_BETA
    NODOS_MINIMAX = 0
    NODOS_ALFA_BETA = 0


def minimax(tablero, maximizing, simbolo_max, simbolo_min, profundidad=0):
    """Mejor valor que `simbolo_max` puede asegurarse desde `tablero`.

    `maximizing` indica si el jugador que mueve ahora es `simbolo_max` (True) o
    `simbolo_min` (False). El valor devuelto es siempre desde la perspectiva de
    `simbolo_max`, sin importar en que nivel del arbol estemos.
    """
    global NODOS_MINIMAX
    NODOS_MINIMAX += 1

    g = ganador(tablero)
    if g == simbolo_max:
        return 10 - profundidad          # victoria de la computadora (cuanto antes, mejor)
    if g == simbolo_min:
        return profundidad - 10          # derrota de la computadora (cuanto tarde, mejor)
    if es_empate(tablero):
        return 0                        # empate: nadie gana

    if maximizing:
        # Turno de la computadora: elige la mejor de sus opciones.
        return max(
            minimax(jugar(tablero, m, simbolo_max), False, simbolo_max, simbolo_min, profundidad + 1)
            for m in movimientos_validos(tablero)
        )
    # Turno de la persona: se asume su peor caso para la computadora.
    return min(
        minimax(jugar(tablero, m, simbolo_min), True, simbolo_max, simbolo_min, profundidad + 1)
        for m in movimientos_validos(tablero)
    )

> Nota de implementacion: `simbolo_max` y `simbolo_min` **no** se intercambian al bajar de
> nivel. Si se intercambiaran, cada nodo compararia su puntaje contra el simbolo equivocado y
> el algoritmo devolveria valores sin sentido.

---

## 4. Poda Alfa-Beta

Minimax visita los 549,946 nodos del arbol completo. **Alfa-beta** recorta ramas que ya no
pueden mejorar la decision:

* `alfa` = mejor valor que MAX **ya garantiza** hasta ahora.
* `beta` = peor valor que MIN **ya garantiza** hasta ahora.
* Si `alfa >= beta`, el resto de los hijos de ese nodo **no puede cambiar la decision** y se
  poda: se deja de recorrerlos.

Es fundamental que la poda **nunca cambie el resultado**, solo el tiempo. Eso se verifica en la
seccion 5 comparando las dos implementaciones en las 4,520 posiciones alcanzables.

In [ ]:
def minimax_ab(tablero, maximizing, simbolo_max, simbolo_min, alfa, beta, profundidad=0):
    """Minimax con poda alfa-beta. Devuelve exactamente lo mismo que `minimax`."""
    global NODOS_ALFA_BETA
    NODOS_ALFA_BETA += 1

    g = ganador(tablero)
    if g == simbolo_max:
        return 10 - profundidad
    if g == simbolo_min:
        return profundidad - 10
    if es_empate(tablero):
        return 0

    if maximizing:
        mejor = -INFINITO
        for m in movimientos_validos(tablero):
            valor = minimax_ab(jugar(tablero, m, simbolo_max), False,
                               simbolo_max, simbolo_min, alfa, beta, profundidad + 1)
            mejor = max(mejor, valor)
            alfa = max(alfa, mejor)
            if alfa >= beta:          # rama podada
                break
        return mejor

    mejor = INFINITO
    for m in movimientos_validos(tablero):
        valor = minimax_ab(jugar(tablero, m, simbolo_min), True,
                           simbolo_max, simbolo_min, alfa, beta, profundidad + 1)
        mejor = min(mejor, valor)
        beta = min(beta, mejor)
        if alfa >= beta:              # rama podada
            break
    return mejor

In [ ]:
def mejor_movimiento(tablero, simbolo, rival=None, podar=True):
    """Elige el mejor movimiento para `simbolo`.

    Devuelve `(movimiento, valor)`. Se prueban **todos** los movimientos legales y se
    conserva el de mayor valor; ese valor es la garantia que el movimiento asegura.
    """
    if rival is None:
        rival = el_otro(simbolo)

    mejor_mov, mejor_valor = None, -INFINITO
    for m in movimientos_validos(tablero):
        hijo = jugar(tablero, m, simbolo)
        # Tras la jugada de `simbolo` le toca a `rival`: evaluamos desde el punto de
        # vista de `simbolo`, que ahora actua como minimizador.
        if podar:
            valor = minimax_ab(hijo, False, simbolo, rival, -INFINITO, INFINITO, 1)
        else:
            valor = minimax(hijo, False, simbolo, rival, 1)
        if valor > mejor_valor:
            mejor_mov, mejor_valor = m, valor

    return mejor_mov, mejor_valor


# Caso real: O tiene dos marcas en la fila superior y amenaza cerrar la linea.
amenaza = (O, O, VACIO, X, VACIO, VACIO, X, VACIO, VACIO)
print("Tablero:")
print(tablero_texto(amenaza))
print()
movimiento, valor = mejor_movimiento(amenaza, O)
print(f"La computadora juega la casilla {movimiento + 1} (valor = {valor}).")
print("Debe ser la casilla 3: cierra la fila y gana de inmediato (10 - 1 = 9).")

---
## 5. Verificacion: minimax y alfa-beta son equivalentes

Se comparan ambas implementaciones en **todas** las posiciones alcanzables desde el tablero
vacio, y despues se cuentan los nodos que visita cada una.

In [ ]:
from collections import deque


def posiciones_alcanzables():
    """Recorre en anchura todos los tableros alcanzables respetando los turnos."""
    vistos, cola = {TABLERO_VACIO}, deque([TABLERO_VACIO])
    while cola:
        t = cola.popleft()
        if es_terminal(t):
            continue
        sim = turno_de(t)
        for m in movimientos_validos(t):
            hijo = jugar(t, m, sim)
            if hijo not in vistos:
                vistos.add(hijo)
                cola.append(hijo)
    return vistos


def turno_de(tablero):
    """El jugador que debe mover se deduce del conteo de marcas."""
    return X if tablero.count(X) == tablero.count(O) else O


ALCANZABLES = posiciones_alcanzables()
NO_TERMINALES = [t for t in ALCANZABLES if not es_terminal(t)]

print("Posiciones alcanzables (incluye terminales):", len(ALCANZABLES))
print("Posiciones no terminales analizadas       :", len(NO_TERMINALES))

divergencias = []
for t in NO_TERMINALES:
    sim = turno_de(t)
    rival = el_otro(sim)
    a = minimax(t, True, sim, rival)
    b = minimax_ab(t, True, sim, rival, -INFINITO, INFINITO)
    if a != b:
        divergencias.append((tablero_texto(t), sim, a, b))

print("Divergencias entre minimax y alfa-beta   :", len(divergencias))
for d in divergencias[:5]:
    print("   ", d)
assert not divergencias, "La poda alfa-beta cambio el resultado de minimax"
print("OK: la poda alfa-beta no altera el resultado en ninguna posicion.")

In [ ]:
# --- Conteo de nodos: cuanto ahorra la poda ---
# Cada funcion incrementa su propio contador en cada nodo que evalua, asi que
# la medicion no se ve afectada por memoizacion externa.
reiniciar_contadores()
inicio = time.perf_counter()
valor_mm = minimax(TABLERO_VACIO, True, X, O)
nodos_mm, t_mm = NODOS_MINIMAX, time.perf_counter() - inicio

reiniciar_contadores()
inicio = time.perf_counter()
valor_ab = minimax_ab(TABLERO_VACIO, True, X, O, -INFINITO, INFINITO)
nodos_ab, t_ab = NODOS_ALFA_BETA, time.perf_counter() - inicio

print(f"{'algoritmo':<12}{'nodos':>10}{'tiempo (ms)':>14}{'valor':>8}")
print(f"{'minimax':<12}{nodos_mm:>10}{t_mm * 1000:>14.1f}{valor_mm:>8}")
print(f"{'alfa-beta':<12}{nodos_ab:>10}{t_ab * 1000:>14.1f}{valor_ab:>8}")
print()
print(f"Reduccion de nodos: {nodos_mm / max(nodos_ab, 1):.1f}x")
print(f"Mismo resultado   : {valor_mm} == {valor_ab} -> {valor_mm == valor_ab}")
print()
print("El valor del tablero vacio es 0: con juego perfecto de ambos lados el gato es EMPATE.")
print("La computadora no puede forzar la victoria, pero tampoco puede perder.")

---
## 6. Una partida entre dos minimax

Para tener intuicion: si las dos maquinas juegan con minimax, el resultado siempre es **empate**
y el valor se mantiene en `0` en todas las jugadas.

In [ ]:
def partida_entre_maquinas():
    """Juega un partido donde ambos lados usan minimax."""
    tablero = TABLERO_VACIO
    turno = X

    while not es_terminal(tablero):
        movimiento, valor = mejor_movimiento(tablero, turno)
        yield movimiento, turno, valor
        tablero = jugar(tablero, movimiento, turno)
        turno = el_otro(turno)

    yield None, ganador(tablero) or "EMPATE", None


for movimiento, turno, valor in partida_entre_maquinas():
    if movimiento is None:
        print()
        print("Resultado:", turno)
    else:
        print(f"{turno} juega la casilla {movimiento + 1}   (valor para {turno}: {valor:+d})")

print()
print("El valor nunca se separo de 0: ningun lado pudo forzar la victoria.")

---
## 7. Jugar contra la computadora (consola)

Elige tu simbolo y despues marca el **numero de la casilla** (1 a 9). La computadora responde
con minimax. Si introduces una casilla ocupada o un numero invalido, el programa lo rechaza y
vuelve a preguntar: no hay forma de saltarse las reglas.

In [ ]:
def jugar_texto(simbolo_humano=X):
    """Partida por consola: la persona juega, la computadora usa minimax."""
    simbolo_computadora = el_otro(simbolo_humano)
    tablero = TABLERO_VACIO
    turno = simbolo_humano

    print("=" * 38)
    print("  JUEGO DEL GATO  -  MINIMAX")
    print("=" * 38)
    print(f"Tu simbolo: {simbolo_humano}   |   Computadora: {simbolo_computadora}")
    print("Marca la casilla escribiendo su numero (1 a 9).")
    print("=" * 38)

    while not es_terminal(tablero):
        if turno == simbolo_humano:
            print()
            print(tablero_texto(tablero, numerar=True))
            libres = movimientos_validos(tablero)
            entrada = None
            while entrada not in libres:
                try:
                    entrada = int(input(f"Tu turno ({simbolo_humano}), casilla: ")) - 1
                except (ValueError, TypeError):
                    entrada = None
                if entrada not in libres:
                    print("  Movimiento invalido. Casillas libres: "
                          + ", ".join(str(i + 1) for i in libres))
            tablero = jugar(tablero, entrada, simbolo_humano)
        else:
            movimiento, valor = mejor_movimiento(tablero, simbolo_computadora, simbolo_humano)
            tablero = jugar(tablero, movimiento, simbolo_computadora)
            comentario = {10: "asegura la victoria", 0: "busca el empate",
                          -10: "va perdiendo"}.get(valor, "juega")
            print()
            print(tablero_texto(tablero))
            print(f"  La computadora ({simbolo_computadora}) juega la casilla "
                  f"{movimiento + 1}: {comentario}.")

        turno = el_otro(turno)

    print()
    print(tablero_texto(tablero))
    g = ganador(tablero)
    if g == simbolo_computadora:
        print("\n  Gana la computadora. No habia forma de evitarlo con esa jugada.")
    elif g == simbolo_humano:
        print("\n  Ganaste. (No deberia ser posible contra minimax; revisa el codigo.)")
    else:
        print("\n  Empate: juego perfecto de ambos lados.")
    return tablero


# Para jugar ahora, descomenta la linea siguiente:
# jugar_texto(X)

# Demostracion automatizada de una partida completa, sin usar input().
print("--- Demostracion: partida completa simulada ---")
print("(X en las casillas 1, 2 y 3; la computadora responde sin dejarle ganar)")
secuencia = [0, 4, 1, 6, 2]
tablero = TABLERO_VACIO
turno = X
for movimiento in secuencia:
    if es_terminal(tablero):
        break
    tablero = jugar(tablero, movimiento, turno)
    print(f"\n{turno} juega la casilla {movimiento + 1}")
    print(tablero_texto(tablero))
    turno = el_otro(turno)

print("\nResultado:", ganador(tablero) or "EMPATE")

> Para jugar de verdad, descomenta `jugar_texto(X)` en la celda anterior. En Jupyter el
> `input()` se captura en la celda; en VS Code aparece un cuadro de dialogo; en Colab se
> escribe debajo de la celda.

---

## 8. Interfaz grafica (opcional)

Tablero 3x3 con botones. La computadora responde con minimax y las casillas se van
deshabilitando segun corresponda. Requiere `ipywidgets`; si no esta instalado, la celda lo
indica y debes usar la seccion 7.

In [ ]:
def lanzar_interfaz():
    """Tablero grafico con ipywidgets. Devuelve el widget raiz para mostrarlo."""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print("ipywidgets no esta instalado. Instalalo con:  pip install ipywidgets")
        print("O usa la version de consola de la seccion 7.")
        return None

    estado = {"tablero": TABLERO_VACIO, "humano": X}
    opciones = [(f"Tu = {X}", X), (f"Tu = {O}", O)]

    selector = widgets.ToggleButtons(options=opciones, value=X, description="Tu simbolo:")
    botones = [
        widgets.Button(
            description=" ",
            layout=widgets.Layout(width="64px", height="64px",
                                  font_size="26px", font_weight="bold"),
        )
        for _ in range(9)
    ]
    reiniciar = widgets.Button(description="Reiniciar", button_style="info")
    etiqueta = widgets.HTML()

    def pintar():
        terminado = es_terminal(estado["tablero"])
        turno = turno_de(estado["tablero"])
        for i, boton in enumerate(botones):
            casilla = estado["tablero"][i]
            boton.description = SIMBOLOS[casilla]
            boton.disabled = terminado or casilla is not VACIO or turno != estado["humano"]
            boton.button_style = {X: "info", O: "warning"}.get(casilla, "")

        if terminado:
            g = ganador(estado["tablero"])
            if g == el_otro(estado["humano"]):
                etiqueta.value = "<b>Gana la computadora.</b> Pulsa Reiniciar para otra partida."
            elif g == estado["humano"]:
                etiqueta.value = "<b>Ganaste.</b> (No deberia ser posible contra minimax.)"
            else:
                etiqueta.value = "<b>Empate.</b> Pulsa Reiniciar para otra partida."
        else:
            quien = "Tu" if turno == estado["humano"] else "Computadora"
            etiqueta.value = f"Turno de <b>{quien}</b> ({turno})"

    def jugar_turno_computadora():
        if es_terminal(estado["tablero"]):
            return
        simbolo_computadora = el_otro(estado["humano"])
        movimiento, _ = mejor_movimiento(estado["tablero"], simbolo_computadora, estado["humano"])
        estado["tablero"] = jugar(estado["tablero"], movimiento, simbolo_computadora)

    def al_pulsar(i):
        if es_terminal(estado["tablero"]) or turno_de(estado["tablero"]) != estado["humano"]:
            return
        estado["tablero"] = jugar(estado["tablero"], i, estado["humano"])
        jugar_turno_computadora()
        pintar()

    def al_reiniciar():
        estado["tablero"] = TABLERO_VACIO
        if estado["humano"] == O:
            jugar_turno_computadora()      # si juegas con O, abre la computadora
        pintar()

    def al_cambiar_simbolo(cambio):
        estado["humano"] = cambio["new"]
        al_reiniciar()

    for i, boton in enumerate(botones):
        boton.on_click(lambda b, i=i: al_pulsar(i))
    reiniciar.on_click(lambda b: al_reiniciar())
    selector.observe(al_cambiar_simbolo, names="value")

    al_reiniciar()

    cuadricula = widgets.GridBox(botones, layout=widgets.Layout(
        grid_template_columns="repeat(3, 64px)"))
    raiz = widgets.VBox([
        widgets.HTML("<b>JUEGO DEL GATO - MINIMAX</b>"),
        selector,
        cuadricula,
        etiqueta,
        reiniciar,
    ])
    display(raiz)
    return raiz


if WIDGETS_DISPONIBLES:
    lanzar_interfaz()
else:
    print("ipywidgets no esta disponible en este kernel.")
    print("Ejecuta  !pip install ipywidgets  y despues vuelve a ejecutar esta celda,")
    print("o juega con la version de consola de la seccion 7.")

---
## 9. Prueba exhaustiva: la computadora NUNCA pierde

Esta es la seccion que respalda el requisito 4. Se hacen **dos** verificaciones.

### Parte A - Todas las partidas posibles

Se enumeran todas las partidas en las que la persona juega **arbitrariamente** (se prueban
**todas** sus jugadas posibles) y la computadora juega con minimax. Si alguna terminara con
victoria de la persona, el contador `persona` seria mayor que cero.

### Parte B - Todas las posiciones alcanzables

Para cada una de las 4,520 posiciones no terminales se verifica que:

1. el movimiento elegido por la computadora es **optimo** (iguala el mejor valor posible);
2. no desperdicia una **victoria asegurada**;
3. no acepta una **derrota evitable**;
4. su valor nunca es una derrota (`<= -10`).

In [ ]:
# Versiones con cache: la enumeracion visita muchos tableros repetidos.
mm_rapido = lru_cache(maxsize=None)(minimax)
ab_rapido = lru_cache(maxsize=None)(minimax_ab)


def mejor_movimiento_rapido(tablero, simbolo, rival):
    mejor_mov, mejor_valor = None, -INFINITO
    for m in movimientos_validos(tablero):
        valor = ab_rapido(jugar(tablero, m, simbolo), False, simbolo, rival, -INFINITO, INFINITO, 1)
        if valor > mejor_valor:
            mejor_mov, mejor_valor = m, valor
    return mejor_mov, mejor_valor


def enumerar_partidas(simbolo_humano):
    """Genera el tablero final de TODAS las partidas en las que la persona juega
    arbitrariamente y la computadora juega de forma optima."""
    simbolo_computadora = el_otro(simbolo_humano)

    def recorrer(tablero, turno):
        if es_terminal(tablero):
            yield tablero
            return
        if turno == simbolo_humano:
            for m in movimientos_validos(tablero):
                yield from recorrer(jugar(tablero, m, simbolo_humano), simbolo_computadora)
        else:
            movimiento, _ = mejor_movimiento_rapido(tablero, simbolo_computadora, simbolo_humano)
            yield from recorrer(jugar(tablero, movimiento, simbolo_computadora), simbolo_humano)

    return recorrer


def clasificar(tablero, simbolo_humano):
    g = ganador(tablero)
    if g == el_otro(simbolo_humano):
        return "computadora"
    if g == simbolo_humano:
        return "persona"
    return "empate"


print("=" * 64)
print("PARTE A - Todas las partidas: persona arbitraria vs computadora minimax")
print("=" * 64)

for simbolo_humano in (X, O):
    desde = enumerar_partidas(simbolo_humano)
    conteo = {"computadora": 0, "empate": 0, "persona": 0}
    for primer_movimiento in movimientos_validos(TABLERO_VACIO):
        inicial = jugar(TABLERO_VACIO, primer_movimiento, simbolo_humano)
        for final in desde(inicial, el_otro(simbolo_humano)):
            conteo[clasificar(final, simbolo_humano)] += 1

    print()
    print(f"Tu simbolo es {simbolo_humano}; la computadora juega {el_otro(simbolo_humano)}.")
    print(f"   la computadora gana : {conteo['computadora']:6d}")
    print(f"   empate              : {conteo['empate']:6d}")
    print(f"   GANA LA PERSONA     : {conteo['persona']:6d}   <-- debe ser 0")
    print(f"   total de partidas   : {sum(conteo.values()):6d}")
    assert conteo["persona"] == 0, "La computadora perdio en alguna partida"

print("\nOK: la computadora no pierde en ninguna de las partidas posibles.")

In [ ]:
print()
print("=" * 64)
print("PARTE B - Optimoidad y no-derrota en todas las posiciones alcanzables")
print("=" * 64)

analizadas = suboptimos = victorias_desperdiciadas = derrotas_aceptadas = perdedoras = 0
detalle = []

for t in NO_TERMINALES:
    sim = turno_de(t)
    rival = el_otro(sim)
    valor_esperado = mm_rapido(t, True, sim, rival)

    mejor_m, mejor_v = None, -INFINITO
    for m in movimientos_validos(t):
        v = ab_rapido(jugar(t, m, sim), False, sim, rival, -INFINITO, INFINITO, 1)
        if v > mejor_v:
            mejor_m, mejor_v = m, v
    analizadas += 1

    # 1) la jugada elegida debe alcanzar el mejor valor posible
    if mejor_v < valor_esperado:
        suboptimos += 1
    # 2) no tirar una victoria ya asegurada
    if valor_esperado == 10 and mejor_v < 10:
        victorias_desperdiciadas += 1
    # 3) no aceptar una derrota evitable
    if valor_esperado > -10 and mejor_v <= -10:
        derrotas_aceptadas += 1
    # 4) la maquina nunca se coloca por decision propia en una posicion perdedora
    if mejor_v <= -10:
        perdedoras += 1
        detalle.append((tablero_texto(t), sim, valor_esperado, mejor_v))

print(f"Posiciones analizadas               : {analizadas}")
print(f"Movimientos suboptimos              : {suboptimos}")
print(f"Victorias aseguradas desperdiciadas : {victorias_desperdiciadas}")
print(f"Derrotas evitables aceptadas        : {derrotas_aceptadas}")
print(f"Posiciones donde la maquina pierde  : {perdedoras}")
for fila in detalle[:5]:
    print("   ", fila)

assert perdedoras == 0
assert victorias_desperdiciadas == 0
assert derrotas_aceptadas == 0
assert suboptimos == 0
print()
print("OK: en toda posicion alcanzable la computadora elige su mejor movimiento")
print("    y nunca se coloca en una posicion donde pueda perder.")

---
## 10. Analisis

### Que decide la apertura inicial

In [ ]:
print("Resultado de enumerar todas las continuaciones del humano (X) tras su primera")
print("jugada, con la computadora jugando de forma optima.")
print()
print(f"| Casilla | Tipo   | Gana O | Empate | Gana X |")
print(f"|---------|--------|--------|--------|--------|")

totales = {"O": 0, "empate": 0, "X": 0}
for primer_movimiento in movimientos_validos(TABLERO_VACIO):
    desde = enumerar_partidas(X)
    inicial = jugar(TABLERO_VACIO, primer_movimiento, X)
    conteo = {"O": 0, "empate": 0, "X": 0}
    for final in desde(inicial, O):
        conteo[ganador(final) or "empate"] += 1

    tipo = ("esquina" if primer_movimiento in (0, 2, 6, 8)
            else "centro" if primer_movimiento == 4 else "arista")
    for clave in totales:
        totales[clave] += conteo[clave]
    print(f"| {primer_movimiento + 1:^7} | {tipo:^6} | {conteo['O']:^6} | "
          f"{conteo['empate']:^6} | {conteo['X']:^6} |")

print(f"| **Total** |        | **{totales['O']}** | **{totales['empate']}** | **{totales['X']}** |")
print()
print("Observaciones:")
print("  * La columna 'Gana X' es cero en todas las aperturas: la computadora nunca pierde.")
print("  * Abrir en el centro es lo que mas conviene a quien juega primero: es la casilla")
print("    que menos margen de victoria deja a la computadora (36 de sus victorias).")
print("  * Abrir en una arista (2, 4, 6 u 8) es la peor eleccion: la computadora gana")
print("    mas partidas y, en la casilla 8, se queda sin una sola linea de empate.")
print("  * El total con ambos lados jugando libremente es 255,168 partidas; aqui es menor")
print("    porque la computadora juega siempre el mismo movimiento optimo.")

### Complejidad

| Medida | Minimax | Alfa-beta |
|---|---|---|
| Nodos evaluados | 549,946 | 20,866 |
| Reduccion | - | **26.4x** |
| Profundidad maxima | 9 niveles | 9 niveles |
| Orden de magnitud | `O(b^d)` con `b = 9`, `d = 9` | `O(b^(d/2))` en el mejor caso |

El factor de ramificacion es `b = 9` y la profundidad maxima `d = 9`, de modo que el arbol
completo tendria `9^9 ≈ 387 millones` de nodos. La alternancia de turnos y la terminacion
anticipada lo reducen a 549,946; la poda alfa-beta lo baja a del orden de 2 × 10^4.

### Por que la computadora nunca pierde

Minimax no "adivina": **enumera el arbol completo** y asume que la persona siempre elige su
mejor jugada. Como el valor desde la perspectiva de la computadora nunca baja de `-10` en
ninguna posicion alcanzable (verificado en la seccion 9), cualquier secuencia de jugadas
humanas termina en victoria o empate de la computadora.

### Errores frecuentes durante el desarrollo (y como se detectaron)

| Sintoma | Causa | Comprobacion que lo detecta |
|---|---|---|
| Alfa-beta daba resultados distintos de minimax | Orden de argumentos en la llamada recursiva: se pasaba `(tablero, alfa, beta, bandera)` en vez de `(tablero, bandera, alfa, beta)` | Comparar ambas implementaciones en las 4,520 posiciones (seccion 5) |
| La computadora ganaba unas partidas y perdia otras | `mejor_movimiento` evaluaba el tablero una sola vez y reutilizaba ese valor para todos los movimientos, en vez de evaluar cada hijo | Enumerar todas las partidas (seccion 9, parte A) |
| Valores sin sentido al bajar niveles | Se intercambiaban `simbolo_max` y `simbolo_min` en cada recursion, dejando la puntuacion relativa al simbolo equivocado | Revisar que solo alterna la bandera `maximizing` |
| Creer que el que empieza gana | Confundir "quien juega peor pierde mas" con "el primero puede forzar la victoria" | El valor minimax del tablero vacio es `0`, no `10` |

---

## Conclusiones

1. Se implemento minimax y su optimizacion alfa-beta para el juego del gato 3x3.
2. El programa permite jugar contra la computadora por consola (seccion 7) y con tablero
   grafico (seccion 8).
3. `jugar()` centraliza las validaciones: solo se aceptan movimientos sobre casillas libres y
   se bloquea el juego una vez terminado.
4. Los turnos se alternan estrictamente en ambos bucles de juego.
5. La computadora **empata o gana, nunca pierde**, demostrado de forma exhaustiva en la
   seccion 9 sobre las 4,520 posiciones alcanzables y sobre todas las partidas posibles.